# **This notebook creates a small, manageable mini-dataset that is a representative microcosm of the full dataset. We will first sample a fixed number of patients (200) while preserving the original class balance (septic vs. non-septic). We will then apply our exact stratified splitting logic to this subset, and optionally create padded versions, allowing us to quickly test our entire modeling pipeline from end to end.**

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import warnings

# =================== Config ===================
# Master source & outputs
DATA_SRC     = Path("/content/drive/MyDrive/Erdos Project Database/all_patients_data_preprocessed.csv")
OUTPUT_ROOT  = Path("/content/drive/MyDrive/Erdos Project Database/outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Column names used throughout
id     = "PatientID"
time   = "Hour"
Sepsis = "SepsisLabel"

# Mini dataset knobs
N_PATIENTS        = 200                       # size of the subset
SPLIT_RATIOS      = np.array([0.6, 0.2, 0.2]) # train, val, test
SEED              = 42
SAVE_CSV_TOO      = True                      # <-- save CSVs for train/val/test as requested

# Optional: also export padded+masked versions for the subset
MAKE_PADDED       = False
PAD_VALUE         = -1.0
LABEL_PAD_VALUE   = -1
PERCENTILE_LEN    = 95                        # pad length = 95th percentile of per-patient length
MAX_LEN_HRS       = None                      # or set a fixed T (int) to override percentile
TRUNCATE_KEEP     = "tail"                    # if truncating > T, keep 'tail' (last T) or 'head' (first T)

# =============== Helpers ===============
def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def load_any(path: Path) -> pd.DataFrame:
    suf = path.suffix.lower()
    if suf in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    elif suf == ".csv":
        return pd.read_csv(path, low_memory=False)
    else:
        raise ValueError(f"Unsupported file type: {suf}")

def save_all(df: pd.DataFrame, out_dir: Path, base: str, save_csv=True, save_parquet=True, save_gzip=True):
    out_dir.mkdir(parents=True, exist_ok=True)
    if save_parquet:
        pq = out_dir / f"{base}.parquet"
        df.to_parquet(pq, index=False); print("Saved:", pq)
    if save_csv:
        cs = out_dir / f"{base}.csv"
        df.to_csv(cs, index=False);     print("Saved:", cs)
    if save_gzip:
        gz = out_dir / f"{base}.csv.gz"
        df.to_csv(gz, index=False, compression="gzip"); print("Saved:", gz)

def exact_counts(n_total: int, ratios: np.ndarray):
    raw = ratios * n_total
    base = np.floor(raw).astype(int)
    rem  = n_total - base.sum()
    if rem > 0:
        frac = raw - base
        order = np.argsort(-frac)
        for j in order[:rem]:
            base[j] += 1
    if base.sum() != n_total:
        base[-1] += (n_total - base.sum())
    return tuple(base.tolist())

def choose_counts_bounded(n_target: int, pos_ratio: float, n_pos_avail: int, n_neg_avail: int):
    n_pos_des, n_neg_des = exact_counts(n_target, np.array([pos_ratio, 1-pos_ratio], float))
    n_pos = min(n_pos_des, n_pos_avail)
    n_neg = min(n_neg_des, n_neg_avail)
    while (n_pos + n_neg) < n_target:
        if n_pos < n_pos_avail: n_pos += 1
        elif n_neg < n_neg_avail: n_neg += 1
        else: break
    n_pos = min(n_pos, n_pos_avail)
    n_neg = min(n_neg, n_neg_avail)
    return n_pos, n_neg

# =============== 1) Import master ===============
assert DATA_SRC.exists(), f"Input file not found: {DATA_SRC}"
df = load_any(DATA_SRC)

print("Loaded master:", DATA_SRC)
print("Shape:", df.shape)
print("First 10 cols:", df.columns[:10].tolist())

# Key columns
assert {id, time}.issubset(df.columns), f"Missing required columns: {id}/{time}"
# Label column: ensure exists and 0/1 int
if Sepsis not in df.columns:
    candidates = [c for c in df.columns if "sepsis" in c.lower()]
    if candidates:
        df[Sepsis] = pd.to_numeric(df[candidates[0]], errors="coerce").fillna(0).clip(0,1).astype(int)
    else:
        df[Sepsis] = 0
df[time]   = pd.to_numeric(df[time], errors="coerce")
df[Sepsis] = pd.to_numeric(df[Sepsis], errors="coerce").fillna(0).clip(0,1).astype(int)
df = df.sort_values([id, time]).reset_index(drop=True)

# Export a clean copy of master (Parquet/CSV/Gzip)
save_all(df, OUTPUT_ROOT, base="master_all_patients", save_csv=True, save_parquet=True, save_gzip=True)

# =============== 2) Build 200-patient subset with preserved septic ratio ===============
subset_dir = ensure_dir(OUTPUT_ROOT / f"subset_{N_PATIENTS}_patients")
splits_dir = ensure_dir(subset_dir / "splits")

# Patient-level labels
has_sepsis = df.groupby(id, sort=False)[Sepsis].max().astype(int)
pids_all   = has_sepsis.index.to_numpy()
y_all      = has_sepsis.to_numpy()
n_total    = len(pids_all)

if n_total < N_PATIENTS:
    warnings.warn(f"Requested {N_PATIENTS} patients but only {n_total} available — using all.")
    N_PATIENTS = n_total

# Global ratio & availability
pos_ratio     = y_all.mean() if n_total else 0.0
pids_pos_all  = pids_all[y_all == 1]
pids_neg_all  = pids_all[y_all == 0]
n_pos_avail   = len(pids_pos_all)
n_neg_avail   = len(pids_neg_all)

# exact bounded counts
n_pos, n_neg = choose_counts_bounded(N_PATIENTS, pos_ratio, n_pos_avail, n_neg_avail)

# sample classwise
rng = np.random.default_rng(SEED)
rng.shuffle(pids_pos_all)
rng.shuffle(pids_neg_all)
pids_pos_sub = set(pids_pos_all[:n_pos])
pids_neg_sub = set(pids_neg_all[:n_neg])
pids_subset  = pids_pos_sub | pids_neg_sub

# Build subset (progress bar over patients for clarity)
mini_chunks = []
for pid in tqdm(sorted(pids_subset), desc="Build 200-patient subset", unit="pt"):
    mini_chunks.append(df[df[id] == pid])
mini = pd.concat(mini_chunks, ignore_index=True)

mini_parq = subset_dir / "subset_full.parquet"
mini.to_parquet(mini_parq, index=False)
if SAVE_CSV_TOO:
    mini.to_csv(subset_dir / "subset_full.csv", index=False)
pd.DataFrame({id: sorted(pids_subset)}).to_csv(subset_dir / "subset_patients.csv", index=False)

print(f"\nSaved mini dataset: {mini_parq}  (patients={len(pids_subset)}, rows={len(mini):,})")

# =============== 3) Exact stratified 60/20/20 split (patient-level) ===============
def split_one_class(pid_list, ratios, rng):
    arr = np.array(sorted(pid_list))
    rng.shuffle(arr)
    n_tr, n_va, n_te = exact_counts(len(arr), ratios)
    return set(arr[:n_tr]), set(arr[n_tr:n_tr+n_va]), set(arr[n_tr+n_va:])

train_pos, val_pos, test_pos = split_one_class(pids_pos_sub, SPLIT_RATIOS, rng)
train_neg, val_neg, test_neg = split_one_class(pids_neg_sub, SPLIT_RATIOS, rng)

train_pids = train_pos | train_neg
val_pids   = val_pos   | val_neg
test_pids  = test_pos  | test_neg

# Sanity: disjoint
assert train_pids.isdisjoint(val_pids)
assert train_pids.isdisjoint(test_pids)
assert val_pids.isdisjoint(test_pids)

# Save split patient lists
pd.DataFrame({id: sorted(train_pids)}).to_csv(splits_dir / "train_patients.csv", index=False)
pd.DataFrame({id: sorted(val_pids)}).to_csv(  splits_dir / "val_patients.csv",   index=False)
pd.DataFrame({id: sorted(test_pids)}).to_csv( splits_dir / "test_patients.csv",  index=False)
print("Saved split patient lists:", splits_dir)

# Save per-split long-format datasets (with a tiny progress bar)
def save_split_df(name: str, pidset: set):
    out = mini[mini[id].isin(pidset)].copy()
    out_parq = subset_dir / f"subset_{name}.parquet"
    out_csv  = subset_dir / f"subset_{name}.csv"
    # Progress bar (dummy for consistency)
    for _ in tqdm(range(1), desc=f"Save {name}", unit="step"):
        out.to_parquet(out_parq, index=False)
        if SAVE_CSV_TOO:
            out.to_csv(out_csv, index=False)
    print(f"Saved {name:>5}: {out_parq}  (patients={len(pidset)}, rows={len(out):,})")
    if SAVE_CSV_TOO:
        print(f"Saved {name:>5}: {out_csv}")

save_split_df("train", train_pids)
save_split_df("val",   val_pids)
save_split_df("test",  test_pids)

# =============== 4) (Optional) padded+masked exports for the subset ===============
if MAKE_PADDED:
    print("\n[Mini Padded] Building padded+masked subset…")
    # numeric features (exclude id/time/label)
    exclude = {id, time, Sepsis}
    num_cols = [c for c in mini.columns if c not in exclude and pd.api.types.is_numeric_dtype(mini[c])]
    for c in num_cols:
        mini[c] = pd.to_numeric(mini[c], errors="coerce")
    mini[time] = pd.to_numeric(mini[time], errors="coerce")

    # target T for subset
    len_per_pid = mini.groupby(id, sort=False).size().rename("len")
    if MAX_LEN_HRS is not None:
        T = int(MAX_LEN_HRS)
    else:
        q = np.percentile(len_per_pid.to_numpy(), PERCENTILE_LEN) if len(len_per_pid) else 0
        T = max(1, int(q))
    print(f"[Mini Padded] Target T = {T} (subset {PERCENTILE_LEN}th percentile)")

    def pad_patient_block(g: pd.DataFrame, T: int, keep: str = "tail"):
        g = g.sort_values(time)
        L = len(g)
        if L > T:
            g = g.iloc[-T:].copy() if keep == "tail" else g.iloc[:T].copy()
            L = T
        steps = np.arange(T, dtype=int)
        mask  = np.zeros(T, dtype=np.int8)
        y     = np.full(T, LABEL_PAD_VALUE, dtype=int)
        X     = np.full((T, len(num_cols)), PAD_VALUE, dtype=float)
        if L > 0:
            mask[:L] = 1
            y[:L] = g[Sepsis].to_numpy(dtype=int, copy=False)
            X[:L, :] = g[num_cols].to_numpy(dtype=float, copy=False)
        out = pd.DataFrame({id: g[id].iloc[0] if L > 0 else np.nan, "Step": steps, "Mask": mask, Sepsis: y})
        for j, c in enumerate(num_cols):
            out[c] = X[:, j]
        return out

    blocks = []
    gb = mini.groupby(id, sort=False)
    for pid, g in tqdm(gb, total=gb.ngroups, unit="pt", desc="[Mini Padded] Pad+Mask", leave=False):
        blocks.append(pad_patient_block(g, T=T, keep=TRUNCATE_KEEP))
    mini_padded = pd.concat(blocks, ignore_index=True)

    pad_dir = ensure_dir(subset_dir / "padded_masked")
    # Progress bar over save steps
    for _ in tqdm(range(1), desc="Save padded full", unit="step"):
        mini_padded.to_parquet(pad_dir / "subset_padded_full.parquet", index=False)
        if SAVE_CSV_TOO:
            mini_padded.to_csv(pad_dir / "subset_padded_full.csv", index=False)

    # attach split label and export per-split padded datasets
    def label_split_col(df_):
        return np.where(df_[id].isin(train_pids), "train",
               np.where(df_[id].isin(val_pids),   "val", "test"))
    mp = mini_padded.copy()
    mp["split"] = label_split_col(mp)

    def save_padded_split(name: str):
        out = mp[mp["split"]==name].drop(columns=["split"])
        out_parq = pad_dir / f"subset_padded_{name}.parquet"
        out_csv  = pad_dir / f"subset_padded_{name}.csv"
        for _ in tqdm(range(1), desc=f"Save padded {name}", unit="step"):
            out.to_parquet(out_parq, index=False)
            if SAVE_CSV_TOO:
                out.to_csv(out_csv, index=False)
        print(f"[Mini Padded] Saved {name}: {out_parq}  (rows={len(out):,})")
        if SAVE_CSV_TOO:
            print(f"[Mini Padded] Saved {name}: {out_csv}")

    save_padded_split("train")
    save_padded_split("val")
    save_padded_split("test")

# =============== 5) Report patient-level proportions ===============
def prop_report(name, pidset):
    hs = has_sepsis.loc[list(pidset)].values
    n = len(hs); pos = int(hs.sum()); neg = n - pos
    p = (pos / n) * 100 if n else 0
    print(f"{name:<5} patients: total={n:4d} | septic={pos:4d} | nonseptic={neg:4d} | septic%={p:5.2f}")

print("\nPatient-level proportions:")
prop_report("GLOBAL", pids_subset)
prop_report("train",  train_pids)
prop_report("val",    val_pids)
prop_report("test",   test_pids)

print("\nDone. Outputs in:", subset_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded master: /content/drive/MyDrive/Erdos Project Database/all_patients_data_preprocessed.csv
Shape: (1552210, 63)
First 10 cols: ['PatientID', 'Hour', 'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2']
Saved: /content/drive/MyDrive/Erdos Project Database/outputs/master_all_patients.parquet
Saved: /content/drive/MyDrive/Erdos Project Database/outputs/master_all_patients.csv
Saved: /content/drive/MyDrive/Erdos Project Database/outputs/master_all_patients.csv.gz


Build 200-patient subset:   0%|          | 0/200 [00:00<?, ?pt/s]


Saved mini dataset: /content/drive/MyDrive/Erdos Project Database/outputs/subset_200_patients/subset_full.parquet  (patients=200, rows=7,712)
Saved split patient lists: /content/drive/MyDrive/Erdos Project Database/outputs/subset_200_patients/splits


Save train:   0%|          | 0/1 [00:00<?, ?step/s]

Saved train: /content/drive/MyDrive/Erdos Project Database/outputs/subset_200_patients/subset_train.parquet  (patients=120, rows=4,697)
Saved train: /content/drive/MyDrive/Erdos Project Database/outputs/subset_200_patients/subset_train.csv


Save val:   0%|          | 0/1 [00:00<?, ?step/s]

Saved   val: /content/drive/MyDrive/Erdos Project Database/outputs/subset_200_patients/subset_val.parquet  (patients=40, rows=1,595)
Saved   val: /content/drive/MyDrive/Erdos Project Database/outputs/subset_200_patients/subset_val.csv


Save test:   0%|          | 0/1 [00:00<?, ?step/s]

Saved  test: /content/drive/MyDrive/Erdos Project Database/outputs/subset_200_patients/subset_test.parquet  (patients=40, rows=1,420)
Saved  test: /content/drive/MyDrive/Erdos Project Database/outputs/subset_200_patients/subset_test.csv

Patient-level proportions:
GLOBAL patients: total= 200 | septic=  15 | nonseptic= 185 | septic%= 7.50
train patients: total= 120 | septic=   9 | nonseptic= 111 | septic%= 7.50
val   patients: total=  40 | septic=   3 | nonseptic=  37 | septic%= 7.50
test  patients: total=  40 | septic=   3 | nonseptic=  37 | septic%= 7.50

Done. Outputs in: /content/drive/MyDrive/Erdos Project Database/outputs/subset_200_patients
